In [1]:
# ============================================================
# CELL 0 — Install libraries
# ============================================================

!pip install -U spacy
!pip install -U scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

  Using cached spacy-3.8.15-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
Using cached spacy-3.8.15-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (32.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 84.4 MB/s eta 0:00:00
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5
    Uninstalling confection-0.1.5:
      Successfully uninstalled confection-0.1.5
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: thinc
    Found existing installation: thinc 8.2.5
    Uninstalling thinc-8.2.5:
      Successfully uninstalled thinc-8.2.5
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.3
    Uninstalling weasel-0

In [2]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# CELL 2 — Imports
# ============================================================

import sys
import json
import time
import importlib.util
from pathlib import Path
from datetime import datetime, timezone

In [4]:
# ============================================================
# CELL 3 — Evaluation Paths
# ============================================================

EVAL_DIR = Path(
    "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/"
    "eval/retrieval_only/retrieval_local"
)

BENCHMARK_DIR = EVAL_DIR / "benchmarks"

RUN_DIR = (
    EVAL_DIR /
    "runs" /
    "local_retrieval_v1"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Evaluation directory:")
print(EVAL_DIR)

print("\nBenchmark directory:")
print(BENCHMARK_DIR)

print("\nRun directory:")
print(RUN_DIR)

Evaluation directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local

Benchmark directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/benchmarks

Run directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1


In [5]:
# ============================================================
# CELL 4 — Locate Benchmark
# ============================================================

benchmark_files = list(
    BENCHMARK_DIR.glob("*.json")
)

print("Benchmark files:")

for path in benchmark_files:
    print(" -", path.name)

preferred = [
    p for p in benchmark_files
    if "aria_local_benchmark_v2" in p.name.lower()
]

if not preferred:
    raise FileNotFoundError(
        "Could not find aria_local_benchmark_v2 JSON "
        f"in {BENCHMARK_DIR}"
    )

BENCHMARK_PATH = preferred[0]

print("\nUsing benchmark:")
print(BENCHMARK_PATH)

Benchmark files:
 - aria_local_benchmark_v2.json

Using benchmark:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/benchmarks/aria_local_benchmark_v2.json


In [6]:
# ============================================================
# CELL 5 — Load Benchmark
# ============================================================

with open(
    BENCHMARK_PATH,
    "r",
    encoding="utf-8"
) as f:

    benchmark = json.load(f)

queries = benchmark["queries"]

print(
    f"Loaded {len(queries)} benchmark queries."
)

print("\nBenchmark metadata:")
print(
    json.dumps(
        benchmark.get("metadata", {}),
        indent=2
    )
)

print("\nQueries:")

for item in queries:
    print(
        f'{item["id"]}: '
        f'{item["query"]}'
    )

Loaded 10 benchmark queries.

Benchmark metadata:
{
  "name": "ARIA Local-Query Benchmark v2.0",
  "description": "Strictly local, entity-centric benchmark for evaluating 1-hop GraphRAG retrieval. Each query has ONE anchor entity. Gold evidence = direct section neighbors of that anchor only (max 10). Designed to test local retrieval, NOT global summarisation.",
  "graph_file": "aria_lite_graph_v2_1_communities_v1.pkl",
  "papers_file": "papers_raw.json",
  "graph_nodes": 2105,
  "graph_edges": 22506,
  "retrieval_type": "local \u2014 1-hop entity neighborhood",
  "gold_evidence_cap": 10
}

Queries:
Q1: What are the cardiovascular risks of aromatase inhibitors compared to tamoxifen in postmenopausal breast cancer patients?
Q2: How are tumor-infiltrating lymphocytes (TILs) scored and used to predict immunotherapy response in breast cancer?
Q3: What are the advances and challenges in HER2-targeted therapy, particularly for neoadjuvant treatment and drug resistance?
Q4: What is the role of

In [7]:
# ============================================================
# CELL 6 — Locate ARIA-Lite Graph
# ============================================================

ARIA_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"
)

GRAPH_NAME = benchmark["metadata"]["graph_file"]

graph_matches = list(
    ARIA_ROOT.rglob(GRAPH_NAME)
)

if not graph_matches:
    raise FileNotFoundError(
        f"Could not find graph file: {GRAPH_NAME}"
    )

GRAPH_PATH = graph_matches[0]

print("Using graph:")
print(GRAPH_PATH)

Using graph:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/aria_lite_graph_v2_1_communities_v1.pkl


In [8]:
# ============================================================
# CELL 7 — Load Graph
# ============================================================

import pickle

with open(
    GRAPH_PATH,
    "rb"
) as f:

    graph = pickle.load(f)

print("Graph loaded.")

print(
    "Nodes:",
    graph.number_of_nodes()
)

print(
    "Edges:",
    graph.number_of_edges()
)

Graph loaded.
Nodes: 2105
Edges: 22506


In [9]:
# ============================================================
# CELL 8 — Locate 6_local_traversal.py
# ============================================================

matches = list(
    ARIA_ROOT.rglob("6_local_traversal.py")
)

if not matches:
    raise FileNotFoundError(
        "Could not find 6_local_traversal.py"
    )

LOCAL_TRAVERSAL_PATH = matches[0]

print(
    "Using local traversal script:"
)

print(
    LOCAL_TRAVERSAL_PATH
)

Using local traversal script:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/src/6_local_traversal.py


In [10]:
# ============================================================
# CELL 9 — Import Existing Retrieval Pipeline
# ============================================================

module_name = "aria_lite_local_traversal"

spec = importlib.util.spec_from_file_location(
    module_name,
    LOCAL_TRAVERSAL_PATH
)

local_traversal_module = (
    importlib.util.module_from_spec(spec)
)

sys.modules[module_name] = (
    local_traversal_module
)

spec.loader.exec_module(
    local_traversal_module
)

print(
    "6_local_traversal.py imported successfully."
)

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


6_local_traversal.py imported successfully.


In [11]:
# ============================================================
# CELL 10 — Get run_local_query()
# ============================================================

if not hasattr(
    local_traversal_module,
    "run_local_query"
):

    raise AttributeError(
        "6_local_traversal.py does not expose "
        "run_local_query()."
    )

run_local_query = (
    local_traversal_module.run_local_query
)

print(
    "run_local_query found:"
)

print(
    run_local_query
)

run_local_query found:
<function run_local_query at 0x7c60213351c0>


In [12]:
# ============================================================
# CELL 11 — Verify Function Signature
# ============================================================

import inspect

print(
    inspect.signature(
        run_local_query
    )
)

(graph, query)


In [13]:
# ============================================================
# CELL 12 — Test Q1
# ============================================================

test_item = queries[0]

print("Query ID:")
print(test_item["id"])

print("\nQuery:")
print(test_item["query"])

start = time.perf_counter()

test_result = run_local_query(
    graph,
    test_item["query"]
)

latency = (
    time.perf_counter() - start
)

print("\nRetrieval completed.")

print(
    f"Latency: {latency:.3f} sec"
)

print(
    "\nResult type:",
    type(test_result)
)

print(
    "\nNumber of retrieved results:",
    len(test_result)
)

Query ID:
Q1

Query:
What are the cardiovascular risks of aromatase inhibitors compared to tamoxifen in postmenopausal breast cancer patients?

Retrieval completed.
Latency: 0.481 sec

Result type: <class 'list'>

Number of retrieved results: 884


In [14]:
# ============================================================
# CELL 13 — Inspect Q1 Result
# ============================================================

print(
    json.dumps(
        test_result[0],
        indent=2,
        default=str
    )
)

{
  "section_id": "42074235_UNLABELLED",
  "score": 172.0,
  "text": "Activating PIK3CA mutations occur in approximately 40% of hormone receptor-positive (HR+)/HER2-negative breast cancers and represent a major driver of endocrine resistance. The PI3K\u03b1-selective inhibitor alpelisib, in combination with fulvestrant, significantly improves progression-free survival in patients with PIK3CA-mutant disease, as demonstrated in the SOLAR-1 trial. However, this therapeutic strategy is frequently complicated by treatment-induced hyperglycemia, a metabolic disturbance that promotes oxidative stress, mitochondrial dysfunction, and inflammatory signaling, thereby increasing cardiovascular vulnerability. Sodium-glucose cotransporter-2 (SGLT2) inhibitors have emerged as cardiometabolic modulators with benefits extending beyond glucose lowering. In this study, we used a human cardiomyocyte in vitro model designed to recapitulate the hyperglycemic metabolic milieu observed in breast cancer patien

In [15]:
# ============================================================
# CELL 14 — Run Full Local Retrieval Benchmark
# ============================================================

all_results = []

print("=" * 80)
print("ARIA-LITE V2 — LOCAL RETRIEVAL EVALUATION")
print("=" * 80)

for i, item in enumerate(
    queries,
    start=1
):

    query_id = item["id"]
    query = item["query"]

    print(
        f"\n[{i}/{len(queries)}] {query_id}"
    )

    start = time.perf_counter()

    ranked_sections = run_local_query(
        graph,
        query
    )

    latency = (
        time.perf_counter() - start
    )

    result_record = {
        "query_id": query_id,
        "query": query,
        "anchor_entity": item.get(
            "anchor_entity"
        ),

        "retrieval": ranked_sections,

        "runtime": {
            "latency_sec": latency,
            "num_results": len(
                ranked_sections
            )
        }
    }

    all_results.append(
        result_record
    )

    print(
        f"Retrieved: "
        f"{len(ranked_sections)} sections"
    )

    print(
        f"Latency: "
        f"{latency:.3f}s"
    )

ARIA-LITE V2 — LOCAL RETRIEVAL EVALUATION

[1/10] Q1
Retrieved: 884 sections
Latency: 0.447s

[2/10] Q2
Retrieved: 947 sections
Latency: 0.911s

[3/10] Q3
Retrieved: 0 sections
Latency: 0.012s

[4/10] Q4
Retrieved: 997 sections
Latency: 1.057s

[5/10] Q5
Retrieved: 666 sections
Latency: 0.124s

[6/10] Q6
Retrieved: 997 sections
Latency: 1.410s

[7/10] Q7
Retrieved: 947 sections
Latency: 1.653s

[8/10] Q8
Retrieved: 966 sections
Latency: 1.939s

[9/10] Q9
Retrieved: 688 sections
Latency: 0.243s

[10/10] Q10
Retrieved: 519 sections
Latency: 0.082s


In [16]:
# ============================================================
# CELL 15 — Save Raw Per-Query Results
# ============================================================

for result in all_results:

    query_id = result["query_id"]

    output_path = (
        RUN_DIR /
        f"{query_id}.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    print(
        "Saved:",
        output_path
    )

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q1.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q2.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q3.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q4.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q5.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q6.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/Q7.json
Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retriev

In [17]:
# ============================================================
# CELL 16 — Save Run Metadata
# ============================================================

run_metadata = {
    "run_name": "local_retrieval_v1",

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "benchmark": str(
        BENCHMARK_PATH
    ),

    "graph": str(
        GRAPH_PATH
    ),

    "retrieval_script": str(
        LOCAL_TRAVERSAL_PATH
    ),

    "retrieval_function": (
        "run_local_query(graph, query)"
    ),

    "num_queries": len(
        queries
    ),

    "notes": (
        "Existing ARIA-Lite v2 local retrieval "
        "pipeline evaluated as-is. No retrieval "
        "logic or traversal configuration was modified."
    )
}

metadata_path = (
    RUN_DIR /
    "run_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    metadata_path
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1/run_metadata.json


In [18]:
# ============================================================
# CELL 17 — Verify Outputs
# ============================================================

print("=" * 80)
print("RAW RETRIEVAL RUN COMPLETE")
print("=" * 80)

output_files = sorted(
    RUN_DIR.glob("*.json")
)

for path in output_files:
    print(" -", path.name)

print(
    f"\nTotal JSON files: "
    f"{len(output_files)}"
)

RAW RETRIEVAL RUN COMPLETE
 - Q1.json
 - Q10.json
 - Q2.json
 - Q3.json
 - Q4.json
 - Q5.json
 - Q6.json
 - Q7.json
 - Q8.json
 - Q9.json
 - run_metadata.json

Total JSON files: 11
